# 🏃 Human Activity Recognition Using CNN + LSTM on KU-HAR Dataset

**Author:** [Your Name]  
**Dataset:** KU-HAR v1.0 (Inertial Measurement Unit signals)  
**Framework:** TensorFlow / Keras  
**Environment:** Google Colab

---

## 🎯 What is this project about?

Human Activity Recognition (HAR) is about teaching a machine to understand what a person is doing — walking, sitting, running, jumping — just by looking at raw sensor signals from an accelerometer and gyroscope.

Think of it like this: when you wear a smartwatch and it says "you walked 5000 steps today", that's HAR in action. The watch is reading tiny vibration signals from sensors and classifying them into activities.

In this project, we'll:
1. Load a real sensor dataset (KU-HAR)
2. Carefully preprocess the signals
3. Apply **Sliding Window** segmentation (a core HAR technique)
4. Apply **Gaussian Window Weighting** (an optional but useful smoothing technique)
5. Build a **CNN + LSTM** deep learning model
6. Train, evaluate, and visualize results

---

## 📌 Why CNN + LSTM?

- **CNN** is great at finding local patterns in signals (like a sharp spike when you jump)
- **LSTM** is great at remembering patterns over time (like a repeated rhythm when you walk)
- Together, they form a powerful pipeline for sequential sensor data

Let's get started! 🚀

---
# 📦 Section 1: Installing Dependencies & Importing Libraries

Before we do anything else, we need to make sure all the required libraries are installed and imported.

We'll need:
- `kagglehub` — to download the dataset directly from Kaggle into Colab
- `pandas`, `numpy` — for data handling
- `matplotlib`, `seaborn` — for visualization
- `sklearn` — for preprocessing and evaluation
- `tensorflow` / `keras` — for building the deep learning model
- `scipy` — for Gaussian window generation

In [ ]:
# First, install kagglehub — this is NOT pre-installed in Colab
# kagglehub is a Python library by Kaggle that lets us download datasets
# directly from Kaggle's servers into Google Colab.
# Without this, we would have to manually upload the dataset every session.

!pip install kagglehub --quiet
print("kagglehub installed successfully!")

kagglehub installed successfully!


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Standard imports — nothing fancy here
# ─────────────────────────────────────────────────────────────────────────────

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scipy gives us the Gaussian window function — we'll use it later
from scipy.signal.windows import gaussian

# Sklearn for preprocessing and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow / Keras for the deep learning model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# Set random seeds so results are reproducible each time we run the notebook
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

All libraries imported successfully!
TensorFlow version: 2.20.0


---
# 📥 Section 2: Loading the KU-HAR Dataset from Kaggle

## ⚠️ Important: Why Kaggle Notebook ≠ Google Colab

If you've ever worked inside Kaggle's own notebook editor, you'll notice that datasets are already available at paths like `/kaggle/input/dataset-name/`. That works because Kaggle automatically mounts your datasets into the notebook's file system.

**But we're in Google Colab.** Colab is a completely different environment — it runs on Google's servers, not Kaggle's. So Kaggle datasets are NOT automatically available here.

To get the dataset into Colab, we use **`kagglehub`** — a library that:
1. Authenticates with your Kaggle account (via your Kaggle API credentials)
2. Downloads the dataset from Kaggle's servers
3. Stores it temporarily in Colab's local storage
4. Returns the local path so we can load the files

**Think of it as a bridge between Kaggle's storage and Colab's workspace.**

> 📌 **Note:** You need to set up Kaggle API credentials in Colab first.  
> Go to: Kaggle → Account → API → Create New Token  
> This gives you a `kaggle.json` file. Upload it to Colab, then run:
> ```python
> from google.colab import files
> files.upload()  # Upload your kaggle.json here
> !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
> ```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Upload your Kaggle API key (kaggle.json)
# Run this ONLY if you haven't set up Kaggle credentials yet
# ─────────────────────────────────────────────────────────────────────────────

# Uncomment the block below to upload your kaggle.json credentials
# ---------------------------------------------------------------
# from google.colab import files
# print("Please upload your kaggle.json file:")
# uploaded = files.upload()
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# print("Kaggle credentials saved!")

print("(Credentials cell — uncomment and run if needed)")

(Credentials cell — uncomment and run if needed)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2: Download the KU-HAR dataset using kagglehub
# ─────────────────────────────────────────────────────────────────────────────

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES
import kagglehub

# This downloads the dataset and returns a path to where it was saved
arashnic_har_1_path = kagglehub.dataset_download('arashnic/har-1')

print('Data source import complete.')
print(f'Dataset saved at: {arashnic_har_1_path}')

100%|██████████| 659M/659M [00:08<00:00, 79.9MB/s]

Extracting files...


Data source import complete.
Dataset saved at: /root/.cache/kagglehub/datasets/arashnic/har-1/versions/2


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3: Construct the full path to the CSV file and load it
# ─────────────────────────────────────────────────────────────────────────────

# The dataset folder has a nested directory structure, so we need to navigate
# into the correct subfolder to find the raw samples CSV.

CSV_PATH = os.path.join(
    arashnic_har_1_path,
    'Time_domain_subsamples',
    '3.Time_domain_subsamples',
    'KU-HAR_v1.0_raw_samples.csv'
)

# Let's verify the file actually exists before loading it
if os.path.exists(CSV_PATH):
    print(f"✅ Found CSV file at: {CSV_PATH}")
else:
    print(f"❌ File not found at: {CSV_PATH}")
    print("Let's see what's inside the downloaded folder...")
    for root, dirs, files in os.walk(arashnic_har_1_path):
        for f in files:
            print(os.path.join(root, f))

✅ Found CSV file at: /root/.cache/kagglehub/datasets/arashnic/har-1/versions/2/Time_domain_subsamples/3.Time_domain_subsamples/KU-HAR_v1.0_raw_samples.csv


In [ ]:
# Loading the CSV into a pandas DataFrame
# This might take a few seconds — the file is fairly large (9185 rows × 9003 columns)

print("Loading dataset... this may take a moment.")
df = pd.read_csv(CSV_PATH)

print(f"\n✅ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"  → {df.shape[0]} samples (rows)")
print(f"  → {df.shape[1]} columns")

## 🧐 Understanding the Dataset Structure

Before we do anything, let's understand what we're working with.

The dataset has **9185 rows** and **9003 columns**. That's a LOT of columns — but it makes sense once you understand the structure:

```
Columns 0–8999  → Sensor signal values (9000 values per sample)
Column 9000     → Activity label (e.g., 'walk', 'run', 'sit')
Column 9001     → Actual signal length (before zero-padding)
Column 9002     → Sample ID
```

### Why 9000 sensor values?

Each sample captures **1500 timesteps** from **6 sensor channels**:
- `ax` — accelerometer X-axis
- `ay` — accelerometer Y-axis  
- `az` — accelerometer Z-axis
- `gx` — gyroscope X-axis
- `gy` — gyroscope Y-axis
- `gz` — gyroscope Z-axis

The data is stored flat: all 1500 values of `ax` first, then all 1500 of `ay`, and so on.

```
[ax_1, ax_2, ..., ax_1500, ay_1, ..., ay_1500, az_1, ..., az_1500, gx_1, ..., gz_1500]
 ←────── 1500 ──────────→ ←─── 1500 ───→ ... (6 channels × 1500 = 9000 total)
```

We'll need to **reshape** this into `(samples, 1500, 6)` for the sequence model.

In [ ]:
# Let's peek at the data to understand it better
print("First few column names (sensor values):")
print(df.columns[:5].tolist(), "...", df.columns[8998:9003].tolist())

print("\nLast 3 columns (metadata):")
print(df.columns[-3:].tolist())

print("\nSample of activity labels (column -3):")
label_col = df.columns[-3]  # activity label is 3rd from last
print(df[label_col].value_counts())

First few column names (sensor values):
['-0.019247', '0.018654', '-0.0005703', '0.013523', '0.013277'] ... ['0.3046', '0.3047', '0.3048', '992', '1']

Last 3 columns (metadata):
['0.3048', '992', '1']

Sample of activity labels (column -3):
0.3048
4     791
7     634
1     596
0     595
3     594
5     588
2     579
11    579
8     553
6     543
16    532
9     512
15    507
14    505
10    414
17    272
12    218
13    172
Name: count, dtype: int64


In [ ]:
# Let's give cleaner names to our key columns for easy reference
# The last 3 columns are: activity label, actual length, sample ID

label_col   = df.columns[-3]   # e.g., 'class'
length_col  = df.columns[-2]   # actual signal length before padding
id_col      = df.columns[-1]   # sample identifier

print(f"Label column    : '{label_col}'")
print(f"Length column   : '{length_col}'")
print(f"ID column       : '{id_col}'")

# How many unique activities are there?
all_activities = df[label_col].unique()
print(f"\nTotal unique activities in dataset: {len(all_activities)}")
print(all_activities)

Label column    : '0.3048'
Length column   : '992'
ID column       : '1'

Total unique activities in dataset: 18
[ 0  1  2  3  4  5  6  7  8 10  9 11 12 13 14 15 16 17]


---
# 🎯 Section 3: Filtering Activities

## Why do we filter activities?

The full KU-HAR dataset contains many activities. For this project, we want to focus on **8 specific activities** that are common in daily life and represent a good variety of motion patterns:

| Activity | Motion Type |
|----------|-------------|
| `stand` | Static / no movement |
| `sit` | Static / no movement |
| `lay` | Static / no movement |
| `stair up` | Locomotion (periodic) |
| `stair down` | Locomotion (periodic) |
| `walk` | Locomotion (periodic) |
| `run` | High-intensity locomotion |
| `jump` | High-intensity intermittent |

Keeping only these helps us:
1. Focus on a well-defined problem
2. Reduce training time
3. Avoid confusion with rare or ambiguous activities
4. Make the classifier more reliable in practice

After filtering, we'll also need to **remap the labels to integers** (0 to 7), because neural networks expect numeric labels, not strings.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define the 8 activities we want to keep
# ─────────────────────────────────────────────────────────────────────────────

target_activities = ['stand', 'sit', 'lay', 'stair up', 'stair down', 'run', 'jump', 'walk']

# Show class distribution BEFORE filtering
print("=" * 50)
print("Class distribution BEFORE filtering:")
print("=" * 50)
print(df[label_col].value_counts())
print(f"\nTotal samples before filtering: {len(df)}")

In [ ]:
# Keep only rows where the label is one of our 8 target activities
df_filtered = df[df[label_col].isin(target_activities)].copy()

# Reset the index — after filtering, the row indices will be scattered.
# Resetting gives us a clean 0, 1, 2, ... index which avoids bugs later.
df_filtered = df_filtered.reset_index(drop=True)

print("=" * 50)
print("Class distribution AFTER filtering:")
print("=" * 50)
print(df_filtered[label_col].value_counts())
print(f"\nTotal samples after filtering: {len(df_filtered)}")

In [ ]:
# Let's visualize the class distribution after filtering
# This is important — if some classes have far fewer samples than others,
# the model might learn to ignore them (class imbalance problem).

plt.figure(figsize=(10, 5))
class_counts = df_filtered[label_col].value_counts()
colors = sns.color_palette('Set2', len(class_counts))

bars = plt.bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.8)

# Add count labels on top of each bar
for bar, count in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
             str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Class Distribution After Activity Filtering', fontsize=14, fontweight='bold')
plt.xlabel('Activity', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Check for imbalance
max_count = class_counts.max()
min_count = class_counts.min()
ratio = max_count / min_count
print(f"\nImbalance ratio (max/min): {ratio:.2f}x")
if ratio > 3:
    print("⚠️  Significant class imbalance detected! Consider oversampling or class weights.")
else:
    print("✅ Class distribution looks reasonably balanced.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Label Encoding
# ─────────────────────────────────────────────────────────────────────────────
# Neural networks can't work with string labels like 'walk' or 'run'.
# We need to convert them to integers: 0, 1, 2, 3, 4, 5, 6, 7
#
# LabelEncoder does this automatically and keeps a record of which number
# maps to which activity — very useful for making predictions readable later.

le = LabelEncoder()
df_filtered['label_encoded'] = le.fit_transform(df_filtered[label_col])

# Show the mapping so we know which number means which activity
print("Label Encoding Map:")
print("-" * 30)
for idx, activity in enumerate(le.classes_):
    print(f"  {idx}  →  {activity}")

print(f"\nNumber of classes: {len(le.classes_)}")

---
# 🔧 Section 4: Separating Features and Labels, Then Reshaping

## Step 1: Separate X (sensor data) and y (labels)

Right now, the sensor data and labels are all in the same DataFrame. We need to:
- Extract the **9000 sensor columns** as our features → `X`
- Extract the **encoded label column** as our target → `y`
- Drop the metadata columns (length and ID) — they're not useful for training

## Step 2: Reshape from 2D to 3D

Currently our data is shaped as `(samples, 9000)` — a flat 2D matrix.

But LSTM and CNN layers in Keras expect **3D input**: `(samples, timesteps, features)`

We need to reshape to `(samples, 1500, 6)` because:
- 1500 = number of timesteps per sample
- 6 = number of sensor channels (ax, ay, az, gx, gy, gz)

The flat data is stored as:
```
[all ax values | all ay values | all az values | all gx values | all gy values | all gz values]
```

After reshape it becomes:
```
timestep_1:  [ax1, ay1, az1, gx1, gy1, gz1]
timestep_2:  [ax2, ay2, az2, gx2, gy2, gz2]
...
timestep_1500: [ax1500, ay1500, az1500, gx1500, gy1500, gz1500]
```

This is the format our sequence model will understand.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Extract the 9000 sensor columns
# The sensor columns are all except the last 3 (label, length, id) and the new encoded label
# ─────────────────────────────────────────────────────────────────────────────

# Get names of all sensor columns (first 9000)
sensor_cols = df_filtered.columns[:9000]
print(f"Number of sensor columns: {len(sensor_cols)}")
print(f"First few: {sensor_cols[:3].tolist()}")
print(f"Last few: {sensor_cols[-3:].tolist()}")

# Extract features and labels
X_flat = df_filtered[sensor_cols].values  # shape: (n_samples, 9000)
y      = df_filtered['label_encoded'].values  # shape: (n_samples,)

print(f"\nX_flat shape: {X_flat.shape}")
print(f"y shape: {y.shape}")
print(f"Unique y values: {np.unique(y)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Reshape from (n_samples, 9000) → (n_samples, 1500, 6)
# ─────────────────────────────────────────────────────────────────────────────

N_TIMESTEPS = 1500  # each sample has 1500 time points
N_CHANNELS  = 6     # ax, ay, az, gx, gy, gz

# The data is currently stored channel-by-channel:
#   [ax_0..ax_1499, ay_0..ay_1499, az_0..az_1499, gx_0..gx_1499, gy_0..gy_1499, gz_0..gz_1499]
#
# We first reshape to (n_samples, 6, 1500) to separate channels,
# then transpose to (n_samples, 1500, 6) to get time-major format.

n_samples = X_flat.shape[0]

# Step 1: Reshape flat → channel-major
X_channel_major = X_flat.reshape(n_samples, N_CHANNELS, N_TIMESTEPS)
print(f"After first reshape (channel-major): {X_channel_major.shape}")

# Step 2: Transpose to time-major (samples, timesteps, channels)
X_3d = X_channel_major.transpose(0, 2, 1)
print(f"After transpose (time-major): {X_3d.shape}")

# Sanity check — let's verify shapes make sense
assert X_3d.shape == (n_samples, N_TIMESTEPS, N_CHANNELS), "Shape mismatch! Check reshape."
print("\n✅ Reshape successful!")
print(f"   Each sample is now: {N_TIMESTEPS} timesteps × {N_CHANNELS} channels")

In [ ]:
# Let's visualize one sample's raw sensor signal to understand the data
# We'll pick a 'walk' sample and look at all 6 channels

channel_names = ['Accel X', 'Accel Y', 'Accel Z', 'Gyro X', 'Gyro Y', 'Gyro Z']
colors_ch = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']

# Find the index of a 'walk' sample
walk_label_idx = list(le.classes_).index('walk')
walk_sample_idx = np.where(y == walk_label_idx)[0][0]  # first walk sample

fig, axes = plt.subplots(6, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'Raw Sensor Signal — Activity: walk (Sample #{walk_sample_idx})',
             fontsize=14, fontweight='bold')

for ch_idx in range(N_CHANNELS):
    axes[ch_idx].plot(X_3d[walk_sample_idx, :, ch_idx],
                      color=colors_ch[ch_idx], linewidth=0.8)
    axes[ch_idx].set_ylabel(channel_names[ch_idx], fontsize=9)
    axes[ch_idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Timestep', fontsize=11)
plt.tight_layout()
plt.show()

print("Notice the repeating pattern in the accelerometer channels — that's the walking rhythm!")

---
# 🪟 Section 5: Windowing — The Heart of HAR Preprocessing

## 🌊 What is Windowing, and Why Does HAR Depend on It?

This is the **most important preprocessing concept** in Human Activity Recognition. Take a moment to really understand it.

---

### The Problem with Raw Continuous Signals

Imagine you're wearing a sensor-equipped watch and walking continuously for 30 seconds. The sensor is recording accelerometer data the entire time — that's potentially **3000+ data points** (at 100Hz sampling rate).

Now imagine trying to classify that entire 30-second signal as one activity. A few problems arise:

1. **Sequences are too long** — LSTMs and CNNs can't easily process arbitrarily long sequences
2. **Activities change** — In the middle of a walk, you might pause or turn, creating messy boundaries
3. **We need multiple training examples** — One 30-second walk gives us only ONE training sample

---

### The Solution: Sliding Windows

Instead of treating the whole signal as one unit, we **slice it into smaller fixed-size chunks** called **windows**.

```
Original signal (1500 timesteps):
──────────────────────────────────────────────────────────────────
t=0                                                           t=1499

Sliding windows (size=200, stride=100, 50% overlap):
Window 1: [0   ──────── 199]
Window 2:       [100 ──────── 299]
Window 3:             [200 ──────── 399]
...
```

Each window becomes **one training example** — a short time-series that captures a snippet of the activity.

---

### Key Windowing Parameters

| Parameter | What it means | Our choice |
|-----------|--------------|------------|
| **Window size** | How many timesteps per window | 200 |
| **Stride** | How many steps to move before next window | 100 |
| **Overlap** | (window_size - stride) / window_size × 100% | 50% |

**Why 50% overlap?**  
- Without overlap, windows might cut right through an important motion (like missing the peak of a jump)
- With 50% overlap, each motion appears in at least two windows, making the model more robust
- It also gives us **more training samples** from the same raw data

**Why window size 200?**  
- At a typical sampling rate of 100Hz, 200 timesteps = 2 seconds of data
- 2 seconds is enough to capture one full gait cycle (walk) or one jump motion
- Too small → not enough context; Too large → captures multiple activities

---

### LSTM Requires Sequential Windows

LSTM (Long Short-Term Memory) networks are designed for **sequential data**. They process input step-by-step and maintain a "memory" of previous steps. For this to work meaningfully, the input must be a short, coherent time window — not an arbitrarily long signal.

So windowing serves a dual purpose: it makes the data manageable AND it makes it suitable for LSTM processing.

---

### Types of Windows in Signal Processing

There are actually several types of windows used in signal processing. They're often confused, so let's clear this up:

| Window Type | Purpose | Used In |
|------------|---------|--------|
| **Sliding Window** | Segment signal into overlapping chunks | HAR, time series classification |
| **Gaussian Window** | Smooth/weight a signal segment | Signal processing, feature weighting |
| **Frequency-domain Window** | Reduce spectral leakage in FFT | Fourier analysis, audio processing |

**Sliding windows** are ESSENTIAL in HAR — they define how we split the data.

**Gaussian windows** are OPTIONAL — they're a weighting function applied INSIDE each sliding window.

These are different things used for different purposes!

In [ ]:
# Let's visualize what sliding windows look like on a real signal

WINDOW_SIZE = 200  # number of timesteps per window
STRIDE = 100       # how many timesteps to skip between windows (50% overlap)

# Use the walk sample we looked at earlier
sample_signal = X_3d[walk_sample_idx, :, 0]  # just the ax channel for visualization

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample_signal, color='gray', linewidth=0.8, label='Raw ax signal', zorder=1)

# Highlight first few windows
window_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for i, start in enumerate(range(0, min(500, len(sample_signal) - WINDOW_SIZE + 1), STRIDE)):
    end = start + WINDOW_SIZE
    ax.axvspan(start, end, alpha=0.25, color=window_colors[i % len(window_colors)],
               label=f'Window {i+1}' if i < 5 else '')

ax.set_title('Sliding Windows on Raw Accelerometer Signal (first 500 timesteps)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Timestep')
ax.set_ylabel('Accel X')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# How many windows would one sample generate?
num_windows = (N_TIMESTEPS - WINDOW_SIZE) // STRIDE + 1
print(f"Window size: {WINDOW_SIZE} timesteps")
print(f"Stride: {STRIDE} timesteps ({int((WINDOW_SIZE-STRIDE)/WINDOW_SIZE*100)}% overlap)")
print(f"Windows per sample: {num_windows}")

---
## 🔔 Gaussian Window Weighting — Why and How

Now let's talk about Gaussian weighting — an optional but powerful technique.

### The Problem with Plain Sliding Windows

When we slice a signal into a window, the edges of that window are **abrupt cut points**. The signal doesn't naturally start and end — we just forced a boundary.

This creates **edge discontinuities**: the signal values at the start and end of the window might not be zero, causing artificial sharp transitions. In signal processing, these discontinuities can introduce **spectral leakage** and **noise artifacts** that confuse the model.

### What is a Gaussian Window?

A Gaussian window is a **bell-shaped weight function** that looks like this:

```
Weight
  1.0 |        ●●●●●
      |      ●●     ●●
  0.5 |    ●●         ●●
      |  ●●             ●●
  0.0 |●●                 ●●
      └────────────────────────
        Start     Center    End
                Timesteps
```

- **Center timesteps** get weight close to 1.0 (full importance)
- **Edge timesteps** get weight close to 0.0 (lower importance)

When we **multiply** each timestep's values by these weights, the edges of the window smoothly fade to zero — eliminating discontinuities.

### Why Might This Help HAR?

1. **Smooths edge discontinuities** — reduces artificial noise at window boundaries
2. **Emphasizes central activity patterns** — the middle of the window is the most "stable" region of the activity
3. **Reduces boundary effects in CNN filters** — convolutional filters at the edges see more relevant data
4. **Previously shown to improve traditional ML models** — empirical evidence from prior work

### The `sigma` Parameter

The Gaussian function is controlled by **sigma (σ)**, the standard deviation:
- **Small σ** → narrow Gaussian → only center timesteps get high weight, edges drop sharply
- **Large σ** → wide Gaussian → most timesteps get roughly equal weight (nearly uniform)

A common choice is `sigma = window_size / 6`, which places ±3σ at the window edges, covering ~99.7% of the bell curve within the window.

In [ ]:
# Let's visualize the Gaussian window and see what it does to a signal segment

# Generate Gaussian weights for our window size
sigma = WINDOW_SIZE / 6  # a common rule of thumb
gauss_weights = gaussian(WINDOW_SIZE, std=sigma)

# Normalize to [0, 1] so the maximum weight is 1
gauss_weights = gauss_weights / gauss_weights.max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: The Gaussian window itself
axes[0].plot(gauss_weights, color='#E74C3C', linewidth=2)
axes[0].fill_between(range(WINDOW_SIZE), gauss_weights, alpha=0.3, color='#E74C3C')
axes[0].set_title(f'Gaussian Window (σ={sigma:.1f})', fontweight='bold')
axes[0].set_xlabel('Timestep within window')
axes[0].set_ylabel('Weight')
axes[0].grid(True, alpha=0.3)

# Plot 2: Original signal window
raw_window = sample_signal[200:400]  # grab a 200-timestep chunk
axes[1].plot(raw_window, color='#3498DB', linewidth=1.5)
axes[1].set_title('Original Signal Window', fontweight='bold')
axes[1].set_xlabel('Timestep within window')
axes[1].set_ylabel('Accel X value')
axes[1].grid(True, alpha=0.3)

# Plot 3: Gaussian-weighted signal window
weighted_window = raw_window * gauss_weights
axes[2].plot(weighted_window, color='#2ECC71', linewidth=1.5)
axes[2].set_title('Gaussian-Weighted Window', fontweight='bold')
axes[2].set_xlabel('Timestep within window')
axes[2].set_ylabel('Weighted value')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Effect of Gaussian Weighting on a Signal Window',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Notice how the edges fade to zero in the Gaussian-weighted version.")
print("The central part of the signal is preserved while boundaries are smoothed out.")

In [ ]:
# Let's also see how different sigma values affect the Gaussian shape
# This helps understand the sigma hyperparameter

sigmas_to_compare = [WINDOW_SIZE / 10, WINDOW_SIZE / 6, WINDOW_SIZE / 4, WINDOW_SIZE / 2]
sigma_labels = ['σ=win/10 (narrow)', 'σ=win/6 (standard)',
                'σ=win/4 (wide)', 'σ=win/2 (very wide)']
colors_sigma = ['#E74C3C', '#F39C12', '#2ECC71', '#3498DB']

plt.figure(figsize=(10, 4))
for s, label, color in zip(sigmas_to_compare, sigma_labels, colors_sigma):
    g = gaussian(WINDOW_SIZE, std=s)
    g = g / g.max()
    plt.plot(g, label=label, color=color, linewidth=2)

plt.title('Gaussian Windows with Different Sigma Values', fontweight='bold', fontsize=13)
plt.xlabel('Timestep within window')
plt.ylabel('Weight')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Narrower σ → More aggressive edge tapering")
print("Wider σ  → Nearly flat weighting (closer to no Gaussian effect)")

---
## 🔨 Implementing Sliding Window + Gaussian Weighting

Now we'll actually implement both windowing steps.

**The pipeline:**
1. For each raw sample (1500 timesteps × 6 channels)
2. Slide a window of size 200 across it with stride 100
3. For each window, multiply by Gaussian weights
4. Collect all windows as new training examples
5. Assign the original label to each window from that sample

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Sliding Window + Gaussian Weighting Implementation
# ─────────────────────────────────────────────────────────────────────────────

WINDOW_SIZE = 200   # window length in timesteps
STRIDE      = 100   # step size between windows (50% overlap)
sigma       = WINDOW_SIZE / 6  # Gaussian standard deviation

# Pre-compute the Gaussian weights — we'll reuse this for every window
gauss_weights = gaussian(WINDOW_SIZE, std=sigma)
gauss_weights = gauss_weights / gauss_weights.max()  # normalize to max = 1

# We'll store all windows and their labels here
windowed_X = []
windowed_y = []

print(f"Processing {X_3d.shape[0]} samples...")
print(f"Window size: {WINDOW_SIZE}, Stride: {STRIDE}")

for sample_idx in range(X_3d.shape[0]):
    signal = X_3d[sample_idx]  # shape: (1500, 6)
    label  = y[sample_idx]

    # Slide the window across the signal
    for start in range(0, N_TIMESTEPS - WINDOW_SIZE + 1, STRIDE):
        end = start + WINDOW_SIZE

        # Extract the window: shape (WINDOW_SIZE, 6)
        window = signal[start:end, :]

        # Apply Gaussian weighting
        # gauss_weights has shape (WINDOW_SIZE,)
        # We need to expand dims to broadcast over 6 channels: (WINDOW_SIZE, 1)
        gauss_column = gauss_weights.reshape(-1, 1)  # shape: (200, 1)
        weighted_window = window * gauss_column       # shape: (200, 6)

        windowed_X.append(weighted_window)
        windowed_y.append(label)

# Convert lists to numpy arrays
windowed_X = np.array(windowed_X)  # shape: (total_windows, WINDOW_SIZE, 6)
windowed_y = np.array(windowed_y)  # shape: (total_windows,)

print(f"\n✅ Windowing complete!")
print(f"   windowed_X shape: {windowed_X.shape}")
print(f"   windowed_y shape: {windowed_y.shape}")
print(f"   Total windows generated: {windowed_X.shape[0]}")
print(f"   Each window: {WINDOW_SIZE} timesteps × {N_CHANNELS} channels")

In [ ]:
# Sanity check: class distribution after windowing
# We want to make sure the class distribution is roughly preserved

print("Class distribution after windowing:")
print("-" * 40)
for class_idx, class_name in enumerate(le.classes_):
    count = np.sum(windowed_y == class_idx)
    print(f"  {class_name:12s}: {count:5d} windows")

print(f"\nTotal windows: {len(windowed_y)}")

# Check for NaN or Inf values that might break training
has_nan = np.isnan(windowed_X).any()
has_inf = np.isinf(windowed_X).any()
print(f"\nNaN values in X: {has_nan}")
print(f"Inf values in X: {has_inf}")
if not has_nan and not has_inf:
    print("✅ Data looks clean — no NaN or Inf values.")

---
# 📐 Section 6: Normalization

## Why Normalize?

Accelerometer values and gyroscope values can have very different scales:
- Accelerometer: typically ±20 m/s²
- Gyroscope: typically ±2000 °/s

If we feed these unscaled values into a neural network, the optimizer will have a hard time because the gradients from gyroscope values will dominate over accelerometer gradients.

**Normalization** brings all values to a similar scale, making training faster and more stable.

We'll use **StandardScaler** which transforms values to have:
- Mean = 0
- Standard deviation = 1

**Important:** We fit the scaler ONLY on training data and then apply it to test data. Fitting on test data would be data leakage.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Train-Test Split BEFORE normalization (to avoid data leakage)
# ─────────────────────────────────────────────────────────────────────────────

# Using stratify=windowed_y ensures both train and test have similar class proportions

X_train, X_test, y_train, y_test = train_test_split(
    windowed_X,
    windowed_y,
    test_size=0.2,
    random_state=42,
    stratify=windowed_y   # important for balanced splits
)

print(f"Train set shape: {X_train.shape}")
print(f"Test  set shape: {X_test.shape}")
print(f"y_train shape:   {y_train.shape}")
print(f"y_test  shape:   {y_test.shape}")

# Verify class proportions are similar
print(f"\nTrain class proportions:")
for i, name in enumerate(le.classes_):
    pct = np.mean(y_train == i) * 100
    print(f"  {name:12s}: {pct:.1f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Normalize the data
# ─────────────────────────────────────────────────────────────────────────────
# The StandardScaler expects 2D input, so we need to temporarily flatten
# our 3D data, fit/transform, then reshape back to 3D.

# Remember shapes:
# X_train: (n_train, WINDOW_SIZE, N_CHANNELS)
# X_test:  (n_test,  WINDOW_SIZE, N_CHANNELS)

n_train = X_train.shape[0]
n_test  = X_test.shape[0]

# Flatten to 2D for scaler: (n_samples, WINDOW_SIZE * N_CHANNELS)
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat  = X_test.reshape(n_test, -1)

# Fit scaler on training data ONLY (never on test data!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled  = scaler.transform(X_test_flat)  # use the same scaler (just transform)

# Reshape back to 3D
X_train_final = X_train_scaled.reshape(n_train, WINDOW_SIZE, N_CHANNELS)
X_test_final  = X_test_scaled.reshape(n_test, WINDOW_SIZE, N_CHANNELS)

print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final  shape: {X_test_final.shape}")

# Quick sanity check
print(f"\nMean of training data (should be ~0): {X_train_final.mean():.4f}")
print(f"Std  of training data (should be ~1): {X_train_final.std():.4f}")

---
# 🧠 Section 7: Building the CNN + LSTM Model

## Understanding the Architecture

Before writing the code, let's understand WHY we're using CNN + LSTM and what each part does.

---

### 🔍 What Does CNN Learn from Sensor Signals?

A **1D Convolutional Neural Network (Conv1D)** works by sliding small filters over the time axis of the signal.

Think of each filter as a "pattern detector". It scans the signal and lights up whenever it finds a pattern that matches what it's learned — like a sharp spike (jump), a periodic oscillation (walk), or a flat region (stand).

Key properties:
- **Local pattern extraction** — each filter looks at a short window (kernel size)
- **Translation invariance** — finds the same pattern whether it appears at timestep 10 or 100
- **Multiple filters** — learns multiple different patterns simultaneously

After convolution, `MaxPooling1D` downsamples the output by keeping only the most prominent activations, reducing sequence length and computation.

---

### 🔁 What Does LSTM Learn?

**Long Short-Term Memory (LSTM)** is a type of Recurrent Neural Network designed to remember patterns over time.

After CNN extracts local features, the sequence is still long enough to have temporal structure. The LSTM reads this feature sequence step-by-step and builds up a "context" — understanding how patterns evolve over time.

For example:
- Walking: CNN detects periodic peaks → LSTM recognizes the repeating rhythmic structure
- Running: CNN detects larger, faster peaks → LSTM recognizes higher-frequency rhythm
- Jumping: CNN detects large impulse spikes → LSTM recognizes the intermittent burst pattern

---

### 🤝 Why CNN + LSTM Together?

| Layer | Responsibility | Input Shape | Output Shape |
|-------|---------------|-------------|--------------|
| Conv1D | Extract local motion patterns | (200, 6) | (200, 64) |
| MaxPool | Reduce & keep prominent features | (200, 64) | (100, 64) |
| Conv1D | Higher-level patterns | (100, 64) | (100, 128) |
| MaxPool | Further reduction | (100, 128) | (50, 128) |
| LSTM | Capture temporal evolution | (50, 128) | (64,) |
| Dense | Classify activity | (64,) | (8,) |

CNN handles the "what" (what kind of motion), LSTM handles the "how it evolves" (is it sustained, rhythmic, or intermittent).

---

### 🆚 Why Deep Learning Instead of Traditional ML?

Traditional ML approaches (SVM, Random Forest) require **manual feature engineering** — you have to explicitly compute features like mean, variance, FFT coefficients, etc.

Deep learning learns features **automatically** from raw data. This is especially powerful for sensor signals because:
- The most useful features aren't always obvious
- Deep models can discover complex temporal patterns humans wouldn't think to engineer

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build the CNN + LSTM Model
# ─────────────────────────────────────────────────────────────────────────────

NUM_CLASSES = len(le.classes_)  # 8 activities
INPUT_SHAPE = (WINDOW_SIZE, N_CHANNELS)  # (200, 6)

print(f"Model input shape: {INPUT_SHAPE}")
print(f"Number of output classes: {NUM_CLASSES}")
print(f"Classes: {list(le.classes_)}")

model = Sequential([

    # ── CNN Block 1 ──────────────────────────────────────────────────────────
    # 64 filters of size 5 → learns 64 different local patterns
    # Each filter looks at 5 consecutive timesteps
    # 'relu' activation introduces non-linearity (learns complex patterns)
    Conv1D(filters=64, kernel_size=5, activation='relu',
           padding='same', input_shape=INPUT_SHAPE),

    # MaxPooling reduces sequence length by 2x
    # Keeps the strongest feature activations, discards weaker ones
    MaxPooling1D(pool_size=2),

    # Dropout randomly turns off 20% of neurons during training
    # This prevents the model from memorizing specific training examples
    Dropout(0.2),

    # ── CNN Block 2 ──────────────────────────────────────────────────────────
    # More filters (128) to learn more complex combinations of basic patterns
    Conv1D(filters=128, kernel_size=5, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    # ── LSTM Layer ───────────────────────────────────────────────────────────
    # After CNN, the sequence has been compressed to (50, 128)
    # LSTM reads this sequence step-by-step and learns temporal dependencies
    # 64 LSTM units — each unit maintains its own memory state
    LSTM(64, return_sequences=False),
    # return_sequences=False means we only keep the final timestep's output
    # (we want a single vector summarizing the whole sequence)

    Dropout(0.3),

    # ── Dense Classification Layers ──────────────────────────────────────────
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Output layer: one neuron per class
    # 'softmax' converts raw scores to probabilities that sum to 1
    Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

## ⚙️ Compiling the Model

Before training, we need to compile the model — specify the optimizer, loss function, and metrics.

### Optimizer: Adam
Adam (Adaptive Moment Estimation) is the most popular optimizer for deep learning.
- It adapts the learning rate for each parameter individually
- Parameters with consistent gradient directions get larger updates
- Parameters with noisy gradients get smaller, more cautious updates
- Essentially, it's a smart version of gradient descent that figures out how fast to learn for each weight

### Loss Function: sparse_categorical_crossentropy
We use `sparse_categorical_crossentropy` (not regular `categorical_crossentropy`) because:
- Our labels are **integers** (0, 1, 2, 3, 4, 5, 6, 7)
- If we used regular `categorical_crossentropy`, we'd need **one-hot encoded** labels ([0,0,1,0,...])
- `sparse_categorical_crossentropy` accepts integer labels directly — saves memory and simplifies code

The cross-entropy loss measures how different the model's predicted probability distribution is from the true label. If the model is very confident and correct → low loss. If the model is confident but wrong → high loss.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compile the model
# ─────────────────────────────────────────────────────────────────────────────

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',   # works with integer labels (not one-hot)
    metrics=['accuracy']
)

print("Model compiled successfully!")
print(f"  Optimizer : Adam")
print(f"  Loss      : sparse_categorical_crossentropy")
print(f"  Metrics   : accuracy")

---
# 🏋️ Section 8: Training the Model

## Key Training Concepts

### Epochs
One epoch = the model sees every training sample exactly once.
We train for multiple epochs because the model needs multiple passes to refine its understanding. Too few epochs → underfitting; too many → overfitting.

### Batch Size
Instead of updating weights after every single sample, we process `batch_size` samples together and then update. This is more computationally efficient and also provides some regularization (the gradient estimates are noisy, which helps escape local minima).

### Validation Split
We reserve 20% of training data for validation — to check how the model performs on data it hasn't trained on, each epoch. This tells us if we're overfitting.

### Early Stopping
We use EarlyStopping to automatically stop training when validation loss stops improving. This prevents overfitting and saves time.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Set up EarlyStopping callback
# ─────────────────────────────────────────────────────────────────────────────

# patience=10 means: if val_loss doesn't improve for 10 consecutive epochs, stop
# restore_best_weights=True means: after stopping, revert to the best epoch's weights

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print("Training the model...")
print(f"  Training samples  : {X_train_final.shape[0]}")
print(f"  Test samples      : {X_test_final.shape[0]}")
print(f"  Input shape       : {X_train_final.shape[1:]}")
print(f"  Classes           : {NUM_CLASSES}")
print("")

history = model.fit(
    X_train_final,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Training complete!")

---
# 📊 Section 9: Training Curves

After training, the `history` object contains a record of loss and accuracy at every epoch for both training and validation data.

Plotting these curves helps us understand:
- **Underfitting**: Both training and validation accuracy are low → model not learning enough
- **Overfitting**: Training accuracy is high but validation accuracy is lower → model memorizing, not generalizing
- **Good fit**: Both curves converge to similarly high accuracy → model is generalizing well

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plot training curves
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = len(history.history['loss'])
epoch_range = range(1, epochs_ran + 1)

# Plot accuracy
axes[0].plot(epoch_range, history.history['accuracy'],
             label='Train Accuracy', color='#3498DB', linewidth=2)
axes[0].plot(epoch_range, history.history['val_accuracy'],
             label='Val Accuracy', color='#E74C3C', linewidth=2, linestyle='--')
axes[0].set_title('Model Accuracy Over Epochs', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot loss
axes[1].plot(epoch_range, history.history['loss'],
             label='Train Loss', color='#3498DB', linewidth=2)
axes[1].plot(epoch_range, history.history['val_loss'],
             label='Val Loss', color='#E74C3C', linewidth=2, linestyle='--')
axes[1].set_title('Model Loss Over Epochs', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_val_acc = max(history.history['val_accuracy'])
best_epoch   = history.history['val_accuracy'].index(best_val_acc) + 1
print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

---
# 🎯 Section 10: Final Evaluation on Test Set

Now we evaluate the trained model on the **held-out test set** — data the model has never seen during training or validation.

We'll look at:
1. **Overall accuracy** — what % of activities were correctly classified
2. **Confusion matrix** — which activities get confused with which others
3. **Classification report** — precision, recall, and F1-score per activity

### Understanding the Metrics

| Metric | What it measures |
|--------|------------------|
| **Precision** | Of all times I predicted 'walk', how many actually were walks? |
| **Recall** | Of all actual 'walk' samples, how many did I correctly find? |
| **F1-score** | Harmonic mean of precision and recall — the balanced score |
| **Support** | How many test samples belong to this class |

**Confusion Matrix:** Each cell (i, j) shows how many samples of class i were predicted as class j. A perfect classifier has only diagonal entries.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluate on test set
# ─────────────────────────────────────────────────────────────────────────────

test_loss, test_accuracy = model.evaluate(X_test_final, y_test, verbose=0)

print("=" * 50)
print("FINAL TEST SET RESULTS")
print("=" * 50)
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Get class predictions for the test set
y_pred_probs = model.predict(X_test_final, verbose=0)   # softmax probabilities
y_pred       = np.argmax(y_pred_probs, axis=1)          # take the class with highest prob

print(f"Predictions shape: {y_pred.shape}")
print(f"First 10 predictions:  {y_pred[:10]}")
print(f"First 10 ground truth: {y_test[:10]}")

# Convert back to activity names for the report
y_pred_names  = le.inverse_transform(y_pred)
y_test_names  = le.inverse_transform(y_test)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Classification Report
# ─────────────────────────────────────────────────────────────────────────────

print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test_names, y_pred_names, target_names=le.classes_))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Confusion Matrix Visualization
# ─────────────────────────────────────────────────────────────────────────────

cm = confusion_matrix(y_test, y_pred)

# Normalize row-wise so each row sums to 1.0 (i.e. recall per class).
# We use np.where to safely handle any row that sums to zero — which would
# happen only if a class is completely absent from the test set, which is
# unlikely after stratified splitting, but it's good defensive practice.
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm  = np.where(row_sums > 0, cm.astype(float) / row_sums, 0.0)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=0.5,
    linecolor='gray'
)

plt.title('Normalized Confusion Matrix\n(CNN + LSTM, KU-HAR Dataset)',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted Activity', fontsize=12)
plt.ylabel('True Activity', fontsize=12)
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nDiagonal = correctly classified (higher is better)")
print("Off-diagonal = misclassifications (lower is better)")
print("\nActivities most commonly confused with each other often have similar motion profiles.")
print("For example, 'stair up' and 'walk' may show some overlap.")


---
# 🚀 Section 11: Bonus — Future Improvements & Research Directions

This section discusses ways to take this project further — useful for viva questions and future work.

## 1. Bidirectional LSTM (BiLSTM)

A regular LSTM reads the sequence from left to right (past to present). A **Bidirectional LSTM** also reads right to left (present to past) and concatenates both outputs. This gives the model context from both directions.

For HAR, the future context of a motion can sometimes clarify ambiguous current states. BiLSTM often improves accuracy by 1-3%.

```python
from tensorflow.keras.layers import Bidirectional
# Replace:  LSTM(64)
# With:     Bidirectional(LSTM(64))
```

## 2. Attention Mechanisms

Not all timesteps in a window are equally important. An attention mechanism lets the model learn which timesteps to focus on. This is inspired by the Transformer architecture.

## 3. Transfer Learning with Physics Toolbox Data

If you've collected your own sensor data using the **Physics Toolbox app** on your phone, you could:
1. Pre-train the model on the large KU-HAR dataset
2. Fine-tune only the final layers on your personal data

This is **transfer learning** — using knowledge from a large dataset to bootstrap learning on a smaller one. Very effective in practice.

## 4. Hyperparameter Tuning Ideas

| Hyperparameter | Current Value | Try Also |
|---------------|---------------|----------|
| Window size | 200 | 100, 300 |
| Stride | 100 | 50, 150 |
| Gaussian sigma | W/6 | W/4, W/8 |
| CNN filters | 64, 128 | 32, 64; 128, 256 |
| LSTM units | 64 | 32, 128 |
| Dropout rate | 0.2, 0.3 | 0.1, 0.4 |
| Batch size | 64 | 32, 128 |
| Learning rate | Adam default | 0.001, 0.0001 |

## 5. Class Imbalance Handling

If some activities have significantly fewer samples:
- Use `class_weight` in `model.fit()` to penalize mistakes on minority classes more
- Use SMOTE (Synthetic Minority Oversampling Technique) to generate synthetic samples
- Use augmentation (add small random noise to signal windows)

## 6. Real-time Inference

For deployment on a smartphone or embedded device:
- Convert model to TFLite: `model.save('model.tflite')`
- Quantize weights to int8 for faster inference
- Connect to live sensor stream from Physics Toolbox

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Optional: BiLSTM version of the model
# Uncomment the entire block below to build and compare against the regular LSTM.
# The Bidirectional import is intentionally kept inside the commented block so
# that this cell doesn't do anything noisy when the BiLSTM is not being used.
# ─────────────────────────────────────────────────────────────────────────────

# from tensorflow.keras.layers import Bidirectional
#
# bilstm_model = Sequential([
#     Conv1D(64, kernel_size=5, activation='relu', padding='same', input_shape=INPUT_SHAPE),
#     MaxPooling1D(pool_size=2),
#     Dropout(0.2),
#     Conv1D(128, kernel_size=5, activation='relu', padding='same'),
#     MaxPooling1D(pool_size=2),
#     Dropout(0.2),
#
#     # BiLSTM reads the sequence in both directions and concatenates the outputs.
#     # This usually gives the model a small but meaningful accuracy boost.
#     Bidirectional(LSTM(64, return_sequences=False)),
#
#     Dropout(0.3),
#     Dense(64, activation='relu'),
#     Dropout(0.3),
#     Dense(NUM_CLASSES, activation='softmax')
# ])
#
# bilstm_model.compile(
#     optimizer='adam',
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )
# bilstm_model.summary()

print("BiLSTM model template ready — uncomment the block above to train and compare!")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Save the trained model (optional but useful)
# ─────────────────────────────────────────────────────────────────────────────

# TensorFlow 2.x still supports .h5, but the recommended modern format is
# the native .keras format (introduced in TF 2.12). It saves everything —
# architecture, weights, optimizer state — in a single clean file.
#
# Uncomment the lines below when you're happy with the model and want to keep it.

# model.save('kuhar_cnn_lstm_model.keras')
# print("Model saved to kuhar_cnn_lstm_model.keras")

# To load it back later in a fresh session:
# from tensorflow.keras.models import load_model
# loaded_model = load_model('kuhar_cnn_lstm_model.keras')
# print("Model loaded successfully!")

print("Model save code is ready — uncomment to save.")


---
# ✅ Section 12: Summary & Takeaways

## What We Did

1. **Loaded** the KU-HAR dataset from Kaggle into Google Colab using `kagglehub`
2. **Understood** the dataset structure (9000 flat sensor values = 1500 timesteps × 6 channels)
3. **Filtered** 8 target activities and encoded labels as integers
4. **Reshaped** data from flat 2D `(n, 9000)` to sequence 3D `(n, 1500, 6)`
5. **Applied Sliding Window segmentation** — split each sample into 200-timestep overlapping windows (50% overlap), generating many more training examples
6. **Applied Gaussian Window Weighting** — multiplied each window by a bell-shaped weight function to smooth edge discontinuities and emphasize central timesteps
7. **Normalized** the data using StandardScaler (fitted only on training data)
8. **Built a CNN + LSTM model** — CNN for local pattern extraction, LSTM for temporal context
9. **Trained** with EarlyStopping to prevent overfitting
10. **Evaluated** using accuracy, confusion matrix, and per-class F1-scores

## Key Concepts to Remember

| Concept | One-line Summary |
|---------|------------------|
| Sliding Windows | Slice long signals into fixed-size training examples |
| Gaussian Weighting | Bell-shaped weights that smooth window edges |
| CNN | Finds local motion patterns (spikes, oscillations) |
| LSTM | Remembers how patterns evolve over time |
| Transfer Learning | Pre-train on big data, fine-tune on yours |
| Label Encoding | Convert string labels to integers for the model |
| StandardScaler | Normalize features to mean=0, std=1 |

---
*Good luck with your project and presentation! 🎓*